# Análise de Dados Climáticos de 2024
## 1. Importar as bibliotecas

As bibliotecas usadas foram pandas e altair.

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 2. Carregar e tratar os dados

In [3]:
file_1 = "dados_climaticos_jardim_botanico.CSV"

def carregar_dados(file, bairro_nome):
    # Ler CSV
    df = pd.read_csv(file, sep=';', decimal=',', encoding='latin1', skiprows=8, header=0)

    # Visualizar primeiras linhas
    print(df.head())

    # Renomear colunas para facilitar uso
    df.columns = [
        "data", "hora", "precipitacao", "pressao",
        "pressao_max", "pressao_min", "radiacao",
        "temperatura", "ponto_orvalho", "temp_max",
        "temp_min", "orvalho_max", "orvalho_min",
        "umidade_max", "umidade_min", "umidade",
        "vento_direcao", "vento_rajada", "vento_velocidade", "extra"
    ]

    # Converter data + hora em datetime
    df['hora'] = df['hora'].str.replace(' UTC', '', regex=False)
    df['datetime'] = pd.to_datetime(df['data'] + ' ' + df['hora'], format='%Y/%m/%d %H%M')

    # Remover colunas antigas e adicionar coluna de bairro
    df = df.drop(columns=['data', 'hora'])
    df['bairro'] = bairro_nome

    # Remover medições completamente nulas
    return df.dropna(how='all')

df = carregar_dados(file_1, "Jardim Botanico")

print(df.shape)

         Data  Hora UTC  PRECIPITAÇÃO TOTAL, HORÁRIO (mm)  \
0  2024/01/01  0000 UTC                               0.0   
1  2024/01/01  0100 UTC                               0.0   
2  2024/01/01  0200 UTC                               0.0   
3  2024/01/01  0300 UTC                               0.0   
4  2024/01/01  0400 UTC                               0.0   

   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  \
0                                             1010.6       
1                                             1011.2       
2                                             1011.1       
3                                             1010.5       
4                                             1010.2       

   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)  \
0                                           1010.6   
1                                           1011.2   
2                                           1011.3   
3                                           1011.1   
4 

## 3. Agregação dos dados


In [4]:
# Conferir estrutura
print(df.info())

# Análise exploratória simples

print(df.describe())

# Convertendo para médias diárias (para suavizar e facilitar visualização)

# 1. Definir um dicionário com a regra de agregação ideal para cada variável relevante
regras_agregacao = {
    'precipitacao': 'sum',          # Queremos o acumulado diário de chuva
    'temperatura': 'mean',          # Temperatura média do dia
    'temp_max': 'max',              # A maior temperatura atingida no dia
    'temp_min': 'min',              # A menor temperatura atingida no dia
    'umidade': 'mean',              # Umidade média do dia
    'umidade_max': 'max',           # Maior umidade do dia
    'umidade_min': 'min',           # Menor umidade do dia
    'vento_velocidade': 'mean',     # Velocidade média do vento
    'vento_rajada': 'max',          # A rajada mais forte do dia
    'pressao': 'mean',              # Pressão atmosférica média
    'radiacao': 'sum'               # Radiação solar acumulada no dia
}

# 2. Aplicar o resample utilizando o dicionário no método .agg()
df_daily = df.resample('D', on='datetime').agg(regras_agregacao).reset_index()

# 3. Se você também utiliza o DataFrame agrupado por Bairro:
df_daily_bairro = df.groupby([
    'bairro', 
    pd.Grouper(key='datetime', freq='D')
]).agg(regras_agregacao).reset_index()

# Criar a coluna de mês para os filtros do Altair (como feito anteriormente)
df_daily['mes'] = df_daily['datetime'].dt.month

<class 'pandas.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   precipitacao      8766 non-null   float64       
 1   pressao           8765 non-null   float64       
 2   pressao_max       8754 non-null   float64       
 3   pressao_min       8754 non-null   float64       
 4   radiacao          4719 non-null   float64       
 5   temperatura       8765 non-null   float64       
 6   ponto_orvalho     8764 non-null   float64       
 7   temp_max          8754 non-null   float64       
 8   temp_min          8754 non-null   float64       
 9   orvalho_max       8754 non-null   float64       
 10  orvalho_min       8753 non-null   float64       
 11  umidade_max       8754 non-null   float64       
 12  umidade_min       8753 non-null   float64       
 13  umidade           8764 non-null   float64       
 14  vento_direcao     8762 non-null   f

## 4. Visualização

In [21]:
# Escalas compartilhadas
xscale = alt.Scale(
    domain=[
        df_daily["temperatura"].min(),
        df_daily["temperatura"].max()
    ]
)

yscale = alt.Scale(
    domain=[
        df_daily["umidade"].min(),
        df_daily["umidade"].max()
    ]
)

# Base
base = (
    alt.Chart(df_daily)
    .transform_calculate(
        log_precipitacao="log(datum.precipitacao + 1) / log(10)"
    )
)

# Scatter principal
scatter = (
    base
    .mark_circle(size=60, opacity=0.8)
    .encode(
        x=alt.X(
            "temperatura:Q",
            title="Temperatura (°C)",
            scale=xscale
        ),
        y=alt.Y(
            "umidade:Q",
            title="Umidade (%)",
            scale=yscale
        ),
        color=alt.Color(
            "log_precipitacao:Q",
            scale=alt.Scale(scheme="turbo"),
            title="Log(Precipitação + 1)",
            legend=alt.Legend(
                orient="bottom",
                direction="horizontal"
            )
        ),
        tooltip=[
            alt.Tooltip("temperatura:Q", title="Temperatura"),
            alt.Tooltip("umidade:Q", title="Umidade"),
            alt.Tooltip("precipitacao:Q", title="Precipitação")
        ]
    )
    .properties(
        width=600,
        height=400
    )
)

# Histograma superior
top_hist = (
    base
    .mark_bar(
        color="#2EC4B6",
        opacity=0.75
    )
    .encode(
        x=alt.X(
            "temperatura:Q",
            bin=alt.Bin(maxbins=30),
            title=None
        ),
        y=alt.Y(
            "count()",
            title="Frequência"
        )
    )
    .properties(
        width=600,
        height=100
    )
)

# Histograma lateral
right_hist = (
    base
    .mark_bar(
        color="#2EC4B6",
        opacity=0.75
    )
    .encode(
        y=alt.Y(
            "umidade:Q",
            bin=alt.Bin(maxbins=30),
            title=None
        ),
        x=alt.X(
            "count()",
            title="Frequência"
        )
    )
    .properties(
        width=120,
        height=400
    )
)

# Layout final
chart = (
    top_hist
    &
    (scatter | right_hist)
).properties(
    title="Padrões Climáticos Observados nos dias de 2024 em Porto Alegre"
).configure_title(
    anchor="middle",
    fontSize=18,
    fontWeight="bold"
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_view(
    stroke=None
)

chart

alt.VConcatChart(...)